In [110]:
!pip install openpyxl pandas matplotlib


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [111]:
import pandas as pd
import matplotlib.pyplot as plt

### Data Overview - 

The focus of this project is on assessing **livability** in Eindhoven whilst population growth pressurizes the capacity of existing infrastructure in the city. **Livability** is defined officially by **Leefbaarometer** as "the extent to which the environment aligns with the demands and wishes placed upon it by people". This definition will be used whenever referencing **livability**.

There are two primary data sources:
- **Leefbaarometer (LF)** - a Dutch monitoring tool (managed by Ministry of Interior and Kingdom Relations) which provides an overview of the liveability of all nieghborhoods and streets in the Netherlands.
    - This data is available from 2002-2020, providing scores at municipality, district, neighborhood, and street level.
    - For this analysis, neighborhood level data is required. 
    - We note from the provided description that some data points are intentionally hidden due to the sample size of residents being insufficient, namely < 100.
    - **LF** is based upon two sub-models - a resident satisfaction model and a housing market model, both of which are used to produce a score for a neighborhood.
- **Onderzoek040 (OZ)** - an open-data and research portal of the Municipality of Eindhoven which includes statistics regarding housing, safety, population, facilities, and social cohesion.
    - These are very similar to the features seen in the LF dataset, making it a valuable point of integration.

In the next sections, we provide an overview of how each dataset was processed independently prior to being integrated holistically. For each dataset, the selected features will be outlined with justification. Any missing data will be handled accordingly at two levels:
- Local To Dataset - each dataset has different reasons for missingness (survey sample size, temporal coverage, administrative changes), which requires different strategies.
- Global After Integration - once data has been integrated into a stable representation, then there will be a second evaluation based on all selected features and the missingness of data.

#### Leefbaarometer Dataset (LF) - 

The LF dataset contains the following columns for all neighborhoods in the Netherlands:
- **Neighborhood Code** - unique identifier for neighborhood.
- **Neighborhood Name** - identifier for neighborhood (possibly not unique).
- **Year** - year of scoring.
- **Livability Score** - the Livability Index Score, which is computed at a grid-level (100x100 meters) from a large number of indicators (~100), some of which include:
    - Perceived nuisance and insecurity.
    - Violent crimes, vandalism, public disorder.
    - Facilities index.
    - Distance to shops, schools, GP, hospital, restaurants, cultural facilities.
    - Population density.
    - Social cohesion.
    - Housing vacancy.
- **Deviation from AVG** - the deviation of the Livability Score from the national average for that year.
- **Physical Environment** - environmental and spatial conditions such as greenery, noise, air quality, flood risk, etc.
- **Nuisance and Insecurity** - safety-related issues such as crime, vandalism, public disorder, etc.
- **Social Cohesion** - how strongly residents are socially connected, familiar with each other, and involved in their locality/neighborhood.
- **Facilities** - access to amenities such as shops, schools, healthcare, public transport, etc.
- **Housing Stock** - characteristics of the homes themselves, such as building age, dwelling size, building type, vacancy, quality, etc.

All of these features are relevant, as they are scores along sub-dimensions (except the overall Livability Score). 

For the **LF** dataset, we need to focus only on the rows with Eindhoven, which has the CBS-BU Code prefix 0772. We note that from our previous pre-processing step, the Leefbaarometer did not exist for all years, and scores are only available for 2002, 2008, 2012, 2014, 2016, 2018, 2020, 2022, and 2024.

In [112]:
lf_df = pd.read_excel('../data/raw/Data Livability.xlsx', header=0)

lf_df = lf_df.rename(columns={
    lf_df.columns[0]: "neighborhood_code",
    lf_df.columns[1]: "neighborhood",
    lf_df.columns[2]: "year",
    lf_df.columns[3]: "livability_score",
    lf_df.columns[4]: "livability_deviation_from_avg",
    lf_df.columns[5]: "physical_environment_deviation",
    lf_df.columns[6]: "nuisance_insecurity_deviation",
    lf_df.columns[7]: "social_cohesion_deviation",
    lf_df.columns[8]: "facilities_deviation",
    lf_df.columns[9]: "housing_stock_deviation",
})

print("Shape: ", lf_df.shape)
print(lf_df.columns)

Shape:  (125105, 10)
Index(['neighborhood_code', 'neighborhood', 'year', 'livability_score',
       'livability_deviation_from_avg', 'physical_environment_deviation',
       'nuisance_insecurity_deviation', 'social_cohesion_deviation',
       'facilities_deviation', 'housing_stock_deviation'],
      dtype='str')


In [113]:
# Keep only Eindhoven data.
lf_df = lf_df[lf_df["neighborhood_code"].str.startswith("BU0772")]

# Drop neighborhood code, no longer useful.
lf_df.drop(columns=["neighborhood_code"], inplace=True)

print("Shape:", lf_df.shape)
print((lf_df.isna().sum() / len(lf_df) * 100).round(1))

Shape: (995, 9)
neighborhood                       0.0
year                               0.0
livability_score                   0.0
livability_deviation_from_avg      0.0
physical_environment_deviation    44.3
nuisance_insecurity_deviation     44.3
social_cohesion_deviation         44.3
facilities_deviation              44.3
housing_stock_deviation           44.3
dtype: float64


In [114]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    lf_df.groupby("year")[["physical_environment_deviation", 
                            "nuisance_insecurity_deviation",
                            "social_cohesion_deviation", 
                            "facilities_deviation",
                            "housing_stock_deviation"]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      physical_environment_deviation  nuisance_insecurity_deviation  social_cohesion_deviation  facilities_deviation  housing_stock_deviation
year                                                                                                                                         
2002                           100.0                          100.0                      100.0                 100.0                    100.0
2008                           100.0                          100.0                      100.0                 100.0                    100.0
2012                           100.0                          100.0                      100.0                 100.0                    100.0
2014                             0.0                            0.0                        0.0                   0.0                      0.0
2016                           100.0                          100.0                      100.0                 100.0                    100.0
2018  

The years 2002, 2008, 2012, and 2016 are structurally absent, excluded from the Leefbaarometer 3.0 dimension scoring. This means that imputation is not a valid option, instead, we should introduce a flag to ensure that our donwstream analysis via knowledge graphs will be aware of the structurally absent nature of this data.

In [115]:
# Years where dimension deviations are structurally absent.
MISSING_DIMENSION_YEARS = [2002, 2008, 2012, 2016]

# Using a boolean mask, flag rows where dimension scores were never computed.
lf_df["dimension_scores_available"] = ~lf_df["year"].isin(MISSING_DIMENSION_YEARS)

print(lf_df.groupby("year")["dimension_scores_available"].first())

year
2002    False
2008    False
2012    False
2014     True
2016    False
2018     True
2020     True
2022     True
2024     True
Name: dimension_scores_available, dtype: bool


#### Facilities Dataset (F) - 

This dataset was sourced from **OZ**, containing 23 indicators measuring the distance in kilometers from each neighborhood to the nearest amenity/public resource of a given type. The categories covered include grocery stores, healthcare, education, leusure, and transport. The distances are computed by CBS through using road networks from each address to the nearest amentiy within the neighborhood.

The dataset was collected from 2015 to 2025, and there are no survey responses, meaning that any detected missingness is administrative or computational as opposed to sample size constraints.

In [116]:
f_df = pd.read_excel('../data/raw/Facilities.xlsx', header=1)

# Renaming the columns for convenience.
f_df = f_df.rename(columns={
    f_df.columns[0]: "neighborhood",
    f_df.columns[1]: "year"
})

# Forward fill so that each row contains a reference to neighborhood.
f_df["neighborhood"] = f_df["neighborhood"].ffill()

# Remove all records where 'year' is not a number.
f_df = f_df[pd.to_numeric(f_df["year"], errors="coerce").notna()]
f_df["year"] = f_df["year"].astype(int)

print("Shape: ", f_df.shape)
print(f_df.columns)

Shape:  (1276, 25)
Index(['neighborhood', 'year', 'Afstand tot grote supermarkt',
       'Afstand tot overige dagelijkse levensmiddelen',
       'Afstand tot attractie', 'Afstand tot bibliotheek',
       'Afstand tot museum', 'Afstand tot podiumkunsten',
       'Afstand tot bioscoop', 'Afstand tot sauna', 'Afstand tot café e.d.',
       'Afstand tot cafetaria e.d.', 'Afstand tot restaurant',
       'Afstand tot zwembad', 'Afstand tot kunstijsbaan',
       'Afstand tot huisartsenpost', 'Afstand tot huisartsenpraktijk',
       'Afstand tot apotheek',
       'Afstand tot ziekenhuis (inclusief buitenpolikliniek)',
       'Afstand tot kinderdagverblijf', 'Afstand tot buitenschoolse opvang',
       'Afstand tot school voor basisonderwijs',
       'Afstand tot school voortgezet onderwijs',
       'Afstand tot oprit hoofdverkeersweg', 'Afstand tot treinstation'],
      dtype='str')


In [117]:
FACILITIES_RENAME = {
    "Afstand tot grote supermarkt":                          "dist_supermarket",
    "Afstand tot overige dagelijkse levensmiddelen":         "dist_daily_groceries",
    "Afstand tot attractie":                                 "dist_attraction",
    "Afstand tot bibliotheek":                               "dist_library",
    "Afstand tot museum":                                    "dist_museum",
    "Afstand tot podiumkunsten":                             "dist_performing_arts",
    "Afstand tot bioscoop":                                  "dist_cinema",
    "Afstand tot sauna":                                     "dist_sauna",
    "Afstand tot café e.d.":                                 "dist_cafe",
    "Afstand tot cafetaria e.d.":                            "dist_cafeteria",
    "Afstand tot restaurant":                                "dist_restaurant",
    "Afstand tot zwembad":                                   "dist_swimming_pool",
    "Afstand tot kunstijsbaan":                              "dist_ice_rink",
    "Afstand tot huisartsenpost":                            "dist_gp_post",
    "Afstand tot huisartsenpraktijk":                        "dist_gp_practice",
    "Afstand tot apotheek":                                  "dist_pharmacy",
    "Afstand tot ziekenhuis (inclusief buitenpolikliniek)":  "dist_hospital",
    "Afstand tot kinderdagverblijf":                         "dist_daycare",
    "Afstand tot buitenschoolse opvang":                     "dist_after_school_care",
    "Afstand tot school voor basisonderwijs":                "dist_primary_school",
    "Afstand tot school voortgezet onderwijs":               "dist_secondary_school",
    "Afstand tot oprit hoofdverkeersweg":                    "dist_highway_ramp",
    "Afstand tot treinstation":                              "dist_train_station",
}

# Renaming columns for convenience.
f_df = f_df.rename(columns=FACILITIES_RENAME)

print((f_df.isna().sum() / len(f_df) * 100).round(1))

neighborhood              0.0
year                      0.0
dist_supermarket          0.0
dist_daily_groceries      0.0
dist_attraction           0.5
dist_library              0.5
dist_museum               0.5
dist_performing_arts      0.5
dist_cinema               0.5
dist_sauna                0.5
dist_cafe                 0.5
dist_cafeteria            0.5
dist_restaurant           0.5
dist_swimming_pool        0.5
dist_ice_rink             0.5
dist_gp_post              0.0
dist_gp_practice          9.1
dist_pharmacy             0.0
dist_hospital             0.0
dist_daycare              9.1
dist_after_school_care    9.1
dist_primary_school       9.1
dist_secondary_school     9.1
dist_highway_ramp         0.0
dist_train_station        0.0
dtype: float64


In [118]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    f_df.groupby("year")[["dist_supermarket",
                           "dist_daily_groceries",
                           "dist_attraction",
                           "dist_library",
                           "dist_museum",
                           "dist_performing_arts",
                           "dist_cinema",
                           "dist_sauna",
                           "dist_cafe",
                           "dist_cafeteria",
                           "dist_restaurant",
                           "dist_swimming_pool",
                           "dist_ice_rink",
                           "dist_gp_post",
                           "dist_gp_practice",
                           "dist_pharmacy",
                           "dist_hospital",
                           "dist_daycare",
                           "dist_after_school_care",
                           "dist_primary_school",
                           "dist_secondary_school",
                           "dist_highway_ramp",
                           "dist_train_station"
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      dist_supermarket  dist_daily_groceries  dist_attraction  dist_library  dist_museum  dist_performing_arts  dist_cinema  dist_sauna  dist_cafe  dist_cafeteria  dist_restaurant  dist_swimming_pool  dist_ice_rink  dist_gp_post  dist_gp_practice  dist_pharmacy  dist_hospital  dist_daycare  dist_after_school_care  dist_primary_school  dist_secondary_school  dist_highway_ramp  dist_train_station
year                                                                                                                                                                                                                                                                                                                                                                                                         
2015               0.0                   0.0              5.2           5.2          5.2                   5.2          5.2         5.2        5.2             5.2              5.2                 5.2     

As shown above, 2015 possesses all the missingness inherent in the dataset, and the percentages are reasonable. This means that we can reasonably impute data from 2016 simply because for public goods or services, it is reasonable to assume that over the course of one year, the distances to facilities remains stable.

In [119]:
# Sort to ensure correct order
f_df = f_df.sort_values(["neighborhood", "year"])

# Filling only the facility distance columns that have 2015 missingness.
facility_cols = [
    "dist_supermarket", "dist_daily_groceries", "dist_attraction",
    "dist_library", "dist_museum", "dist_performing_arts", "dist_cinema",
    "dist_sauna", "dist_cafe", "dist_cafeteria", "dist_restaurant",
    "dist_swimming_pool", "dist_ice_rink", "dist_gp_post", "dist_gp_practice",
    "dist_pharmacy", "dist_hospital", "dist_daycare", "dist_after_school_care",
    "dist_primary_school", "dist_secondary_school", "dist_highway_ramp",
    "dist_train_station"
]

for col in facility_cols:
    f_df[col] = f_df.groupby("neighborhood")[col].bfill()

print(f_df.columns.tolist())
print((f_df.isna().sum() / len(f_df) * 100).round(1))

['neighborhood', 'year', 'dist_supermarket', 'dist_daily_groceries', 'dist_attraction', 'dist_library', 'dist_museum', 'dist_performing_arts', 'dist_cinema', 'dist_sauna', 'dist_cafe', 'dist_cafeteria', 'dist_restaurant', 'dist_swimming_pool', 'dist_ice_rink', 'dist_gp_post', 'dist_gp_practice', 'dist_pharmacy', 'dist_hospital', 'dist_daycare', 'dist_after_school_care', 'dist_primary_school', 'dist_secondary_school', 'dist_highway_ramp', 'dist_train_station']
neighborhood              0.0
year                      0.0
dist_supermarket          0.0
dist_daily_groceries      0.0
dist_attraction           0.0
dist_library              0.0
dist_museum               0.0
dist_performing_arts      0.0
dist_cinema               0.0
dist_sauna                0.0
dist_cafe                 0.0
dist_cafeteria            0.0
dist_restaurant           0.0
dist_swimming_pool        0.0
dist_ice_rink             0.0
dist_gp_post              0.0
dist_gp_practice          0.0
dist_pharmacy             

In [120]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    f_df.groupby("year")[["dist_supermarket",
                           "dist_daily_groceries",
                           "dist_attraction",
                           "dist_library",
                           "dist_museum",
                           "dist_performing_arts",
                           "dist_cinema",
                           "dist_sauna",
                           "dist_cafe",
                           "dist_cafeteria",
                           "dist_restaurant",
                           "dist_swimming_pool",
                           "dist_ice_rink",
                           "dist_gp_post",
                           "dist_gp_practice",
                           "dist_pharmacy",
                           "dist_hospital",
                           "dist_daycare",
                           "dist_after_school_care",
                           "dist_primary_school",
                           "dist_secondary_school",
                           "dist_highway_ramp",
                           "dist_train_station"
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      dist_supermarket  dist_daily_groceries  dist_attraction  dist_library  dist_museum  dist_performing_arts  dist_cinema  dist_sauna  dist_cafe  dist_cafeteria  dist_restaurant  dist_swimming_pool  dist_ice_rink  dist_gp_post  dist_gp_practice  dist_pharmacy  dist_hospital  dist_daycare  dist_after_school_care  dist_primary_school  dist_secondary_school  dist_highway_ramp  dist_train_station
year                                                                                                                                                                                                                                                                                                                                                                                                         
2015               0.0                   0.0              0.0           0.0          0.0                   0.0          0.0         0.0        0.0             0.0              0.0                 0.0     

#### Housing Stock Dataset (H) - 

This dataset was sourced from **OZ**, covering 116 Eindhoven neighborhoods annually from 2014 to 2025, focusing on 6 indicators derived from the Basic Registration of Addresses and Buildings (BAG) and municipal records, capturing hte composition and value of the housing stock in each neighborhood. It also does not contain a survey component.

In [121]:
h_df = pd.read_excel('../data/raw/Housing Stock.xlsx', header=1)

# Renaming the columns for convenience.
h_df = h_df.rename(columns={
    h_df.columns[0]: "neighborhood",
    h_df.columns[1]: "year"
})

# Forward fill so that each row contains a reference to neighborhood.
h_df["neighborhood"] = h_df["neighborhood"].ffill()

# Remove all records where 'year' is not a number.
h_df = h_df[pd.to_numeric(h_df["year"], errors="coerce").notna()]
h_df["year"] = h_df["year"].astype(int)

print("Shape: ", h_df.shape)
print(h_df.columns)

Shape:  (1392, 8)
Index(['neighborhood', 'year', 'Huurwoningen', 'Koopwoningen',
       'Corporatiewoningen', 'Woningen', 'Aandeel woningen bewoond',
       'Woz-waarde'],
      dtype='str')


In [122]:
HOUSING_RENAME = {
    "Huurwoningen":             "rental_dwellings",
    "Koopwoningen":             "owner_occupied_dwellings",
    "Corporatiewoningen":       "corporation_dwellings",
    "Woningen":                 "total_dwellings",
    "Aandeel woningen bewoond": "pct_occupied",
    "Woz-waarde":               "avg_woz_value",
}

h_df = h_df.rename(columns=HOUSING_RENAME)

print((h_df.isna().sum() / len(h_df) * 100).round(1))

neighborhood                0.0
year                        0.0
rental_dwellings            0.0
owner_occupied_dwellings    0.0
corporation_dwellings       0.0
total_dwellings             0.0
pct_occupied                0.0
avg_woz_value               8.3
dtype: float64


In [123]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    h_df.groupby("year")[["rental_dwellings",
                           "owner_occupied_dwellings",
                           "corporation_dwellings",
                           "total_dwellings",
                           "pct_occupied",
                           "avg_woz_value",
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      rental_dwellings  owner_occupied_dwellings  corporation_dwellings  total_dwellings  pct_occupied  avg_woz_value
year                                                                                                                 
2014               0.0                       0.0                    0.0              0.0           0.0            0.0
2015               0.0                       0.0                    0.0              0.0           0.0            0.0
2016               0.0                       0.0                    0.0              0.0           0.0            0.0
2017               0.0                       0.0                    0.0              0.0           0.0            0.0
2018               0.0                       0.0                    0.0              0.0           0.0            0.0
2019               0.0                       0.0                    0.0              0.0           0.0            0.0
2020               0.0                       0.0        

As shown, only 2025 is missing the value estimations, and in this case, forward fill is a reasonable choice as we can assume the property values at worst remain the same, when general trends suggest that they improve.

TODO: Could we get a market appreciation rate?

In [124]:
h_df = h_df.sort_values(["neighborhood", "year"])

h_df["avg_woz_value"] = h_df.groupby("neighborhood")["avg_woz_value"].ffill()

# We create an indicator for WOZ missingness in 2014-2015.
WOZ_MISSING_YEARS = [2014, 2015]
h_df["woz_available"] = ~h_df["year"].isin(WOZ_MISSING_YEARS)

print((h_df.isna().sum() / len(h_df) * 100).round(1))

neighborhood                0.0
year                        0.0
rental_dwellings            0.0
owner_occupied_dwellings    0.0
corporation_dwellings       0.0
total_dwellings             0.0
pct_occupied                0.0
avg_woz_value               0.0
woz_available               0.0
dtype: float64


#### Nuisance and Safety Dataset (NS) - 

This dataset is sourced from **OZ** containing 12 indicators split across two distinct categories:
- pct_feels_unsafe, pct_perceives_crime, safety_rating - based on survey indicators collected through resident surveys.
- The remaining are administrative indicators derived through the share of residents reporting various forms of noise disturbance or pollution.

In [125]:
ns_df = pd.read_excel('../data/raw/Nuisance and Safety.xlsx', header=1)

# Renaming the columns for convenience.
ns_df = ns_df.rename(columns={
    ns_df.columns[0]: "neighborhood",
    ns_df.columns[1]: "year"
})

# Forward fill so that each row contains a reference to neighborhood.
ns_df["neighborhood"] = ns_df["neighborhood"].ffill()

# Remove all records where 'year' is not a number.
ns_df = ns_df[pd.to_numeric(ns_df["year"], errors="coerce").notna()]
ns_df["year"] = ns_df["year"].astype(int)

print("Shape: ", ns_df.shape)
print(ns_df.columns)

Shape:  (1392, 14)
Index(['neighborhood', 'year',
       'Voelt zich wel eens onveilig in de eigen buurt',
       'Aandeel personen dat de perceptie heeft dat er veel criminaliteit in de eigen buurt is',
       'Rapportcijfer dat mensen geven aan veiligheid in hun buurt',
       'Heeft last van geluidshinder veroorzaakt door vliegverkeer',
       'Heeft last van geluidshinder veroorzaakt door treinverkeer',
       'Heeft last van geluidshinder veroorzaakt door bedrijven en horeca',
       'Heeft last van geluidshinder veroorzaakt door wegverkeer',
       'Heeft last van geluidshinder bij evenementen',
       'Heeft last van geluidshinder door buurtbewoners',
       'Heeft last van vervuilde lucht', 'Heeft last van stank',
       'Heeft last van (minstens 1 vorm van) geluidshinder'],
      dtype='str')


In [126]:
NUISANCE_RENAME = {
    "Voelt zich wel eens onveilig in de eigen buurt":                                        "pct_feels_unsafe",
    "Aandeel personen dat de perceptie heeft dat er veel criminaliteit in de eigen buurt is": "pct_perceives_crime",
    "Rapportcijfer dat mensen geven aan veiligheid in hun buurt":                            "safety_rating",
    "Heeft last van geluidshinder veroorzaakt door vliegverkeer":                            "pct_noise_air_traffic",
    "Heeft last van geluidshinder veroorzaakt door treinverkeer":                            "pct_noise_rail_traffic",
    "Heeft last van geluidshinder veroorzaakt door bedrijven en horeca":                     "pct_noise_business",
    "Heeft last van geluidshinder veroorzaakt door wegverkeer":                              "pct_noise_road_traffic",
    "Heeft last van geluidshinder bij evenementen":                                          "pct_noise_events",
    "Heeft last van geluidshinder door buurtbewoners":                                       "pct_noise_neighbors",
    "Heeft last van vervuilde lucht":                                                        "pct_polluted_air",
    "Heeft last van stank":                                                                  "pct_bad_smell",
    "Heeft last van (minstens 1 vorm van) geluidshinder":                                    "pct_any_noise",
}

ns_df = ns_df.rename(columns=NUISANCE_RENAME)

print((ns_df.isna().sum() / len(ns_df) * 100).round(1))

neighborhood               0.0
year                       0.0
pct_feels_unsafe          33.3
pct_perceives_crime       33.3
safety_rating             33.3
pct_noise_air_traffic      0.0
pct_noise_rail_traffic     0.0
pct_noise_business         0.0
pct_noise_road_traffic     0.0
pct_noise_events           0.0
pct_noise_neighbors        0.0
pct_polluted_air           0.0
pct_bad_smell              0.0
pct_any_noise              0.0
dtype: float64


In [127]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    ns_df.groupby("year")[["pct_feels_unsafe",
                           "pct_perceives_crime",
                           "safety_rating",
                           "pct_noise_air_traffic",
                           "pct_noise_rail_traffic",
                           "pct_noise_business",
                           "pct_noise_road_traffic",
                           "pct_noise_events",
                           "pct_noise_neighbors",
                           "pct_polluted_air",
                           "pct_bad_smell",
                           "pct_any_noise"
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      pct_feels_unsafe  pct_perceives_crime  safety_rating  pct_noise_air_traffic  pct_noise_rail_traffic  pct_noise_business  pct_noise_road_traffic  pct_noise_events  pct_noise_neighbors  pct_polluted_air  pct_bad_smell  pct_any_noise
year                                                                                                                                                                                                                                        
2014             100.0                100.0          100.0                    0.0                     0.0                 0.0                     0.0               0.0                  0.0               0.0            0.0            0.0
2015             100.0                100.0          100.0                    0.0                     0.0                 0.0                     0.0               0.0                  0.0               0.0            0.0            0.0
2016             100.0                100.0         

We see a missingness pattern from 2014-2017, as the municipal resident survey was not administered at neighborhood level in those years, or the samples were too small to publish. Backward fill is not a good choice here as it would misrepresent four years of resident experience, which is roughly 33% of data for those columns. Perceived safety is not structurally stable, and can vary year-to-year based on several factors. Hence, we will simply flag whether the survey was available in these years.

In [128]:
MISSING_SURVEY_YEARS = [2014, 2015, 2016, 2017]

ns_df["survey_available"] = ~ns_df["year"].isin(MISSING_SURVEY_YEARS)

print((ns_df.isna().sum() / len(ns_df) * 100).round(1))

neighborhood               0.0
year                       0.0
pct_feels_unsafe          33.3
pct_perceives_crime       33.3
safety_rating             33.3
pct_noise_air_traffic      0.0
pct_noise_rail_traffic     0.0
pct_noise_business         0.0
pct_noise_road_traffic     0.0
pct_noise_events           0.0
pct_noise_neighbors        0.0
pct_polluted_air           0.0
pct_bad_smell              0.0
pct_any_noise              0.0
survey_available           0.0
dtype: float64


#### Physical Environment Dataset (PE) - 

This dataset is sourced from **OZ**, containing 5 indicators derived from CBS land-use statistics, capturing characteristics of each neighborhood such as address density (number of addresses per km^2), urnanity class (a CBS 1-5 categorical scale where 1 is strongly urban, and 5 is not urban).

In [129]:
pe_df = pd.read_excel('../data/raw/Physical Environment.xlsx', header=1)

# Renaming the columns for convenience.
pe_df = pe_df.rename(columns={
    pe_df.columns[0]: "neighborhood",
    pe_df.columns[1]: "year"
})

# Forward fill so that each row contains a reference to neighborhood.
pe_df["neighborhood"] = pe_df["neighborhood"].ffill()

# Remove all records where 'year' is not a number.
pe_df = pe_df[pd.to_numeric(pe_df["year"], errors="coerce").notna()]
pe_df["year"] = pe_df["year"].astype(int)

print("Shape: ", pe_df.shape)
print(pe_df.columns)

Shape:  (1392, 7)
Index(['neighborhood', 'year', 'Omgevingsadressendichtheid', 'Stedelijkheid',
       'Oppervlakte water (in ha)', 'Oppervlakte land (in ha)',
       'Oppervlakte totaal (in ha)'],
      dtype='str')


In [130]:
PHYSICAL_RENAME = {
    "Omgevingsadressendichtheid": "address_density",
    "Stedelijkheid":              "urbanity_class",
    "Oppervlakte water (in ha)":  "area_water_ha",
    "Oppervlakte land (in ha)":   "area_land_ha",
    "Oppervlakte totaal (in ha)": "area_total_ha",
}

pe_df = pe_df.rename(columns=PHYSICAL_RENAME)

print((pe_df.isna().sum() / len(pe_df) * 100).round(1))

neighborhood       0.0
year               0.0
address_density    0.0
urbanity_class     0.0
area_water_ha      0.0
area_land_ha       0.0
area_total_ha      0.0
dtype: float64


In [131]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    pe_df.groupby("year")[["address_density",
                           "urbanity_class",
                           "area_water_ha",
                           "area_land_ha",
                           "area_total_ha",
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      address_density  urbanity_class  area_water_ha  area_land_ha  area_total_ha
year                                                                             
2014              0.0             0.0            0.0           0.0            0.0
2015              0.0             0.0            0.0           0.0            0.0
2016              0.0             0.0            0.0           0.0            0.0
2017              0.0             0.0            0.0           0.0            0.0
2018              0.0             0.0            0.0           0.0            0.0
2019              0.0             0.0            0.0           0.0            0.0
2020              0.0             0.0            0.0           0.0            0.0
2021              0.0             0.0            0.0           0.0            0.0
2022              0.0             0.0            0.0           0.0            0.0
2023              0.0             0.0            0.0           0.0            0.0
2024            

No missingness shows how the data is structurally sound.

#### Social Cohesion Dataset (SC) - 

This dataset is sourced from **OZ** containing four indicators measuring the degree to which residents feel connected to and actively participate in their neighborhood. All indicators are derived from a resident survey.

In [132]:
sc_df = pd.read_excel('../data/raw/Social Cohesion.xlsx', header=1)

# Renaming the columns for convenience.
sc_df = sc_df.rename(columns={
    sc_df.columns[0]: "neighborhood",
    sc_df.columns[1]: "year"
})

# Forward fill so that each row contains a reference to neighborhood.
sc_df["neighborhood"] = sc_df["neighborhood"].ffill()

# Remove all records where 'year' is not a number.
sc_df = sc_df[pd.to_numeric(sc_df["year"], errors="coerce").notna()]
sc_df["year"] = sc_df["year"].astype(int)

print("Shape: ", sc_df.shape)
print(sc_df.columns)

Shape:  (928, 6)
Index(['neighborhood', 'year', 'Sociale cohesie schaalscore',
       '% heeft zich het afgelopen jaar actief ingezet voor de buurt',
       '% wil zich in nabije toekomst zeker actief (blijven) inzetten voor de buurt?',
       '% wil zich in nabije toekomst misschien actief (blijven) inzetten voor de buurt?'],
      dtype='str')


In [133]:
COHESION_RENAME = {
    "Sociale cohesie schaalscore":                                                          "cohesion_scale_score",
    "% heeft zich het afgelopen jaar actief ingezet voor de buurt":                         "pct_actively_engaged",
    "% wil zich in nabije toekomst zeker actief (blijven) inzetten voor de buurt?":         "pct_wants_to_engage",
    "% wil zich in nabije toekomst misschien actief (blijven) inzetten voor de buurt?":     "pct_maybe_engage",
}

sc_df = sc_df.rename(columns=COHESION_RENAME)

print((sc_df.isna().sum() / len(sc_df) * 100).round(1))

neighborhood             0.0
year                     0.0
cohesion_scale_score     0.0
pct_actively_engaged    62.5
pct_wants_to_engage     62.5
pct_maybe_engage        62.5
dtype: float64


In [134]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    sc_df.groupby("year")[["cohesion_scale_score",
                           "pct_actively_engaged",
                           "pct_wants_to_engage",
                           "pct_maybe_engage",
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      cohesion_scale_score  pct_actively_engaged  pct_wants_to_engage  pct_maybe_engage
year                                                                                   
2018                   0.0                 100.0                100.0             100.0
2019                   0.0                 100.0                100.0             100.0
2020                   0.0                 100.0                100.0             100.0
2021                   0.0                 100.0                100.0             100.0
2022                   0.0                 100.0                100.0             100.0
2023                   0.0                   0.0                  0.0               0.0
2024                   0.0                   0.0                  0.0               0.0
2025                   0.0                   0.0                  0.0               0.0


The missingness percentage is very high given the voluntary nature of the surveys, where the cohesion_scale_score is the only column available entirely. This means that a similar flagging strategy will be applied.

In [135]:
MISSING_ENGAGEMENT_YEARS = [2018, 2019, 2020, 2021, 2022]

sc_df["engagement_available"] = ~sc_df["year"].isin(MISSING_ENGAGEMENT_YEARS)

print((sc_df.isna().sum() / len(sc_df) * 100).round(1))

neighborhood             0.0
year                     0.0
cohesion_scale_score     0.0
pct_actively_engaged    62.5
pct_wants_to_engage     62.5
pct_maybe_engage        62.5
engagement_available     0.0
dtype: float64


#### Inhabitants Dataset (I) -

This dataset is sourced from **OZ**, which differs from previous datasets as it has years as columns as opposed to rows, requiring a different transformation. 

In [136]:
i_df = pd.read_excel('../data/raw/Inhabitants.xlsx', header=1)

# Renaming the columns for convenience.
i_df = i_df.rename(columns={
    i_df.columns[0]: "neighborhood",
})

# We expand each row into multiple, each with a (neighborhood, year) structure.
i_df = i_df.melt(
    id_vars="neighborhood",
    var_name="year",
    value_name="population"
)
i_df["year"] = i_df["year"].astype(int)

print("Shape: ", i_df.shape)
print(i_df.columns)

Shape:  (2457, 3)
Index(['neighborhood', 'year', 'population'], dtype='str')


In [137]:
print((i_df.isna().sum() / len(i_df) * 100).round(1))

neighborhood    0.0
year            0.0
population      0.7
dtype: float64


In [138]:
# Collecting missing data by year, normalizing by all neighborhoods in that year.
missing_by_year = (
    i_df.groupby("year")[["population",
                        ]]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
)
print(missing_by_year.to_string())

      population
year            
2005         0.9
2006         0.9
2007         0.9
2008         0.9
2009         0.9
2010         0.9
2011         0.9
2012         0.9
2013         0.9
2014         0.9
2015         0.9
2016         0.9
2017         0.9
2018         0.9
2019         0.0
2020         0.0
2021         0.0
2022         0.0
2023         0.0
2024         0.9
2025         0.9


In [139]:
# Find neighborhoods that are missing population in any year
missing_neighborhoods = (
    i_df[i_df["population"].isna()]
    .groupby("neighborhood")["year"]
    .apply(list)
    .reset_index()
)
missing_neighborhoods.columns = ["neighborhood", "years_missing"]
print(missing_neighborhoods.to_string(index=False))

neighborhood                                                                                    years_missing
    Onbekend [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2024, 2025]


Onbekend translates to "unknown", which means that this missingness does not incidate any structural issues in the dataset.

### Integration - 

All datasets share the (neighborhood, year) as the join key, so the integration is a straightforward chain of merge operations. 

In [140]:
from functools import reduce

dfs = [lf_df, f_df, h_df, ns_df, pe_df, sc_df, i_df]

# Correcting UTF-8 encoding problem with Castiliëlaan.
for df in [lf_df, f_df, h_df, ns_df, pe_df, sc_df, i_df]:
    df["neighborhood"] = df["neighborhood"].str.replace("Ă«", "ë", regex=False)

merged = reduce(lambda left, right: pd.merge(
    left, right, on=["neighborhood", "year"], how="outer"
), dfs)

print("Shape: ", merged.shape)
pd.set_option('display.max_rows', None)
print((merged.isna().sum() / len(merged) * 100).round(1))
pd.reset_option('display.max_rows')

Shape:  (2567, 64)
neighborhood                       0.0
year                               0.0
livability_score                  61.2
livability_deviation_from_avg     61.2
physical_environment_deviation    78.4
nuisance_insecurity_deviation     78.4
social_cohesion_deviation         78.4
facilities_deviation              78.4
housing_stock_deviation           78.4
dimension_scores_available        61.2
dist_supermarket                  50.3
dist_daily_groceries              50.3
dist_attraction                   50.3
dist_library                      50.3
dist_museum                       50.3
dist_performing_arts              50.3
dist_cinema                       50.3
dist_sauna                        50.3
dist_cafe                         50.3
dist_cafeteria                    50.3
dist_restaurant                   50.3
dist_swimming_pool                50.3
dist_ice_rink                     50.3
dist_gp_post                      50.3
dist_gp_practice                  50.3
dist_p

The high degree of missingness in the data above has three primary reasons:
- **Temporal Coverage Mismatch** - the outer join spans 2002–2025 (~21 years) but most Onderzoek040 sources only start at 2014/2015. 
    - Every row before 2014 will have NaN for all Onderzoek040 columns - this accounts for the baseline ~45.9% missingness across housing, physical environment, and noise columns.
- **Leefbaarometer Measurement Years** - Leefbaarometer only has 9 snapshot years, not annual data.
    - Hence, livability_score is absent for all non-measurement years.
- **Survey-Driven Sparsity** - The perception and engagement columns are absent for years the survey wasn't administered.
    - This has already been flagged.

Based on this, we will formulate our analysis from 2014 onwards, which should improve the outlook of our data missingness.

In [141]:
# After the outer merge, the created flags are NaN for years where no Leefbaarometer observation existed.
# We need to ensure that we fill missing values with False here to keep flag integrity.
merged["dimension_scores_available"] = merged["dimension_scores_available"].fillna(False)
merged["survey_available"] = merged["survey_available"].fillna(False)
merged["engagement_available"] = merged["engagement_available"].fillna(False)

print(merged.groupby("year")[["dimension_scores_available", 
                               "survey_available", 
                               "engagement_available"]].first().to_string())

     dimension_scores_available survey_available engagement_available
year                                                                 
2002                      False            False                False
2005                      False            False                False
2006                      False            False                False
2007                      False            False                False
2008                      False            False                False
2009                      False            False                False
2010                      False            False                False
2011                      False            False                False
2012                      False            False                False
2013                      False            False                False
2014                       True            False                False
2015                      False            False                False
2016                

In [142]:
merged["woz_available"] = merged["woz_available"].fillna(False)

In [143]:
merged = merged[merged["year"].between(2014, 2025)]

pd.set_option('display.max_rows', None)
print((merged.isna().sum() / len(merged) * 100).round(1))
pd.reset_option('display.max_rows')

neighborhood                       0.0
year                               0.0
livability_score                  52.6
livability_deviation_from_avg     52.6
physical_environment_deviation    60.5
nuisance_insecurity_deviation     60.5
social_cohesion_deviation         60.5
facilities_deviation              60.5
housing_stock_deviation           60.5
dimension_scores_available         0.0
dist_supermarket                   9.1
dist_daily_groceries               9.1
dist_attraction                    9.1
dist_library                       9.1
dist_museum                        9.1
dist_performing_arts               9.1
dist_cinema                        9.1
dist_sauna                         9.1
dist_cafe                          9.1
dist_cafeteria                     9.1
dist_restaurant                    9.1
dist_swimming_pool                 9.1
dist_ice_rink                      9.1
dist_gp_post                       9.1
dist_gp_practice                   9.1
dist_pharmacy            

In [144]:
# Dropping "Unknown" neighborhood, which is an administrative placeholder.
merged = merged[merged["neighborhood"] != "Onbekend"]

print(merged["neighborhood"].nunique())

116


At this point, we are not going to remove missing data by row missingness. This is because, in consultation with Group 7, we can still make useful aggregations of different neighborhoods in our analysis to maximize our data usage.

In [145]:
# Some final adjustments such as removing missingness indicators and translation.

URBANITY_TRANSLATE = {
    "Zeer sterk stedelijk": "Very strongly urban",
    "Sterk stedelijk":      "Strongly urban",
    "Matig stedelijk":      "Moderately urban",
    "Weinig stedelijk":     "Slightly urban",
    "Niet stedelijk":       "Not urban",
}

merged["urbanity_class"] = merged["urbanity_class"].map(
    URBANITY_TRANSLATE
).fillna(merged["urbanity_class"])

df = df.replace('.', pd.NA)

df = df.apply(lambda col: col.map(lambda x: pd.NA if str(x).strip() == '.' else x))

In [146]:
merged.to_csv("../data/processed/merged.csv")